In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving tiny-shakespeare.txt to tiny-shakespeare.txt


In [ ]:
with open ("tiny-shakespeare.txt","r",encoding="UTF-8") as f:
    text=f.read()

In [ ]:
len(text)

1115394

In [ ]:
text[:1000:]

"First Citizen:\nBefore we proceed any further, hear me speak.\n\nAll:\nSpeak, speak.\n\nFirst Citizen:\nYou are all resolved rather to die than to famish?\n\nAll:\nResolved. resolved.\n\nFirst Citizen:\nFirst, you know Caius Marcius is chief enemy to the people.\n\nAll:\nWe know't, we know't.\n\nFirst Citizen:\nLet us kill him, and we'll have corn at our own price.\nIs't a verdict?\n\nAll:\nNo more talking on't; let it be done: away, away!\n\nSecond Citizen:\nOne word, good citizens.\n\nFirst Citizen:\nWe are accounted poor citizens, the patricians good.\nWhat authority surfeits on would relieve us: if they\nwould yield us but the superfluity, while it were\nwholesome, we might guess they relieved us humanely;\nbut they think we are too dear: the leanness that\nafflicts us, the object of our misery, is as an\ninventory to particularise their abundance; our\nsufferance is a gain to them Let us revenge this with\nour pikes, ere we become rakes: for the gods know I\nspeak this in hunger 

In [ ]:
import re
text=text.lower()
text=re.sub(r"\n+"," ",text)

In [ ]:
len(text)

1108171

In [ ]:
text[:1000:]

"first citizen: before we proceed any further, hear me speak. all: speak, speak. first citizen: you are all resolved rather to die than to famish? all: resolved. resolved. first citizen: first, you know caius marcius is chief enemy to the people. all: we know't, we know't. first citizen: let us kill him, and we'll have corn at our own price. is't a verdict? all: no more talking on't; let it be done: away, away! second citizen: one word, good citizens. first citizen: we are accounted poor citizens, the patricians good. what authority surfeits on would relieve us: if they would yield us but the superfluity, while it were wholesome, we might guess they relieved us humanely; but they think we are too dear: the leanness that afflicts us, the object of our misery, is as an inventory to particularise their abundance; our sufferance is a gain to them let us revenge this with our pikes, ere we become rakes: for the gods know i speak this in hunger for bread, not in thirst for revenge. second ci

In [ ]:
vocab=sorted(set(text))

In [ ]:
vocab_len=len(vocab)

In [ ]:
char_to_index = {}
for idx, ch in enumerate(vocab):
    char_to_index[ch] = idx

In [ ]:
index_to_char = {idx: ch for idx, ch in enumerate(vocab)}


In [ ]:
encoded_text = []
for ch in text:
    encoded_text.append(char_to_index[ch])

In [ ]:
seq_length = 100

sequences = []
targets = []

for i in range(len(encoded_text) - seq_length):
    sequences.append(encoded_text[i : i + seq_length])
    targets.append(encoded_text[i + seq_length])

In [ ]:
import torch
import torch.nn as nn

from torch.utils.data import TensorDataset, DataLoader

X_tensor = torch.tensor(sequences, dtype=torch.long)
y_tensor = torch.tensor(targets, dtype=torch.long)

dataset = TensorDataset(X_tensor, y_tensor)
dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

In [ ]:
class LSTM(nn.Module):
    def __init__(self,vocab_len,input_size,hidden_size,num_layers,dropout=0.2):
        super().__init__()

        self.input_size=input_size
        self.hidden_size=hidden_size
        self.num_layers=num_layers


        self.embedding=nn.Embedding(vocab_len,self.input_size)
        self.lstm=nn.LSTM(
            self.input_size,
            self.hidden_size,
            self.num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc=nn.Linear(self.hidden_size,vocab_len)

    def forward(self, x, hidden=None):
        x = self.embedding(x)
        out, hidden = self.lstm(x, hidden)
        out = out[:, -1, :]
        out = self.fc(out)
        return out, hidden

In [ ]:
import torch

# Check if the T4 GPU is available, otherwise default to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}") # Make sure this outputs 'cuda'

# Instantiate the model
model = LSTM(vocab_len, input_size=100, hidden_size=256, num_layers=2)

# PUSH THE MODEL TO THE GPU
model = model.to(device)

import torch.optim as optim
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

Using device: cuda


In [ ]:
train_size=int(0.8*len(dataset))
test_size=len(dataset)-train_size

train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=256,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
    drop_last=True

)

In [ ]:
epochs=20

for epoch in range(epochs):
    model.train()
    epoch_loss=0.0

    for xb,yb in train_loader:
        xb = xb.to(device)
        yb = yb.to(device)
        optimizer.zero_grad()

        output, _=model(xb)
        loss=criterion(output,yb)
        loss.backward()
        optimizer.step()
        epoch_loss+=loss.item()

    print(f"epoch = {epoch+1} & loss = {epoch_loss/len(train_loader)}")




epoch = 1 & loss = 1.7029309704219724
epoch = 2 & loss = 1.4267459110949234
epoch = 3 & loss = 1.360824309698355
epoch = 4 & loss = 1.3235762257027806
epoch = 5 & loss = 1.2978782209270197
epoch = 6 & loss = 1.280105217525134
epoch = 7 & loss = 1.2656019597401997
epoch = 8 & loss = 1.2539293531012081
epoch = 9 & loss = 1.243982606182754
epoch = 10 & loss = 1.2361693411703953
epoch = 11 & loss = 1.228850569616915
epoch = 12 & loss = 1.222531432812094
epoch = 13 & loss = 1.2172855726044554
epoch = 14 & loss = 1.2125402256149147
epoch = 15 & loss = 1.207994154359067
epoch = 16 & loss = 1.204320870266699
epoch = 17 & loss = 1.2016132893863956
epoch = 18 & loss = 1.19795545083398
epoch = 19 & loss = 1.1964328474801513
epoch = 20 & loss = 1.1932412669157306


In [ ]:
import math

model.eval()
with torch.no_grad():
    corr_val = 0
    total_val = 0
    total_loss = 0.0

    for xb, yb in test_loader:

        xb = xb.to(device)
        yb = yb.to(device)
        output, _ = model(xb)
        loss = criterion(output, yb)

        preds = output.argmax(dim=1)
        total_val += yb.size(0)
        corr_val += (preds == yb).sum().item()
        total_loss += loss.item()

avg_loss = total_loss / len(test_loader)
perplexity = math.exp(avg_loss)

print(f"accuracy = {corr_val/total_val*100}%")
print(f"perplexity = {perplexity}")

accuracy = 60.037030346820806%
perplexity = 3.566667014195093


In [ ]:
import torch
print(torch.cuda.is_available())

True


In [ ]:
import torch

# Save only the model weights (the most efficient way)
torch.save(model.state_dict(), 'shakespeare_lstm1.pth')
print("Model successfully saved to shakespeare_lstm.pth!")

Model successfully saved to shakespeare_lstm.pth!


In [ ]:
from google.colab import files

# This will trigger a browser download for the file
files.download('shakespeare_lstm1.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>